# SCOPE Core Call Graph

This notebook statically inspects the Python files in `core/src/scope`. It does not import or execute SCOPE. The generated interactive map provides a module overview, searchable function-level neighborhoods, and source locations for resolved calls.

Python permits dynamic dispatch, runtime imports, aliases, and monkey patching. Consequently, this is an architectural aid rather than a guaranteed runtime trace. Calls that cannot be resolved safely are retained in the exported JSON and shown in the node details.

In [7]:
import ast
import json
import os
import subprocess
from collections import Counter, defaultdict
from datetime import datetime

from IPython.display import IFrame, display

def find_scope_repository(start_path):
    current_path = os.path.abspath(start_path)
    while True:
        if os.path.isdir(current_path + "/core/src/scope"):
            return current_path
        parent_path = os.path.dirname(current_path)
        if parent_path == current_path:
            raise FileNotFoundError("Could not locate the SCOPE repository from the current working directory.")
        current_path = parent_path


scope_repo_path = find_scope_repository(os.getcwd())
scope_core_path = scope_repo_path + "/core/src/scope"
output_path     = scope_repo_path + "/docs/development/scope_core_graph"
html_output     = output_path + "/scope_core_graph.html"
json_output     = output_path + "/scope_core_graph.json"

if not os.path.isdir(scope_core_path):
    raise FileNotFoundError(f"SCOPE core package not found: {scope_core_path}")
os.makedirs(output_path, exist_ok=True)

## 1. Parse modules and definitions

In [8]:
def get_module_name(filepath):
    relative = os.path.relpath(filepath, os.path.dirname(scope_core_path))
    parts = relative[:-3].split(os.sep)
    if parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


python_files = []
for root, _, filenames in os.walk(scope_core_path):
    for filename in filenames:
        if filename.endswith(".py"):
            python_files.append(os.path.join(root, filename))
python_files.sort()

module_data = {}
for filepath in python_files:
    with open(filepath, encoding="utf-8") as file:
        source = file.read()
    module = get_module_name(filepath)
    module_data[module] = {
        "module": module,
        "filepath": filepath,
        "relative_file": os.path.relpath(filepath, os.path.dirname(scope_core_path)),
        "is_package": os.path.basename(filepath) == "__init__.py",
        "source": source,
        "tree": ast.parse(source, filename=filepath),
    }


definitions = {}
definition_asts = {}
classes = {}


class DefinitionVisitor(ast.NodeVisitor):
    def __init__(self, module):
        self.module = module
        self.context = []
        self.function_depth = 0

    def visit_ClassDef(self, node):
        class_qualname = self.module + "." + ".".join([name for kind, name in self.context] + [node.name])
        classes[class_qualname] = {
            "id": class_qualname,
            "name": node.name,
            "module": self.module,
            "bases": [ast.unparse(base) for base in node.bases],
            "file": module_data[self.module]["relative_file"],
            "line": node.lineno,
        }
        self.context.append(("class", node.name))
        self.generic_visit(node)
        self.context.pop()

    def _visit_function(self, node, is_async=False):
        names = [name for _, name in self.context] + [node.name]
        function_id = self.module + "." + ".".join(names)
        class_names = [name for kind, name in self.context if kind == "class"]
        owner_class = None
        if class_names and self.function_depth == 0:
            owner_class = self.module + "." + ".".join(class_names)
        if self.function_depth > 0:
            kind = "nested"
        elif owner_class:
            kind = "method"
        else:
            kind = "function"
        docstring = ast.get_docstring(node) or ""
        definitions[function_id] = {
            "id": function_id,
            "name": node.name,
            "label": (owner_class.rsplit(".", 1)[-1] + "." + node.name) if owner_class else node.name,
            "module": self.module,
            "owner_class": owner_class,
            "kind": "async_" + kind if is_async else kind,
            "file": module_data[self.module]["relative_file"],
            "line": node.lineno,
            "end_line": getattr(node, "end_lineno", node.lineno),
            "private": node.name.startswith("_") and not (node.name.startswith("__") and node.name.endswith("__")),
            "dunder": node.name.startswith("__") and node.name.endswith("__"),
            "doc": docstring.splitlines()[0] if docstring else "",
        }
        definition_asts[function_id] = node
        self.context.append(("function", node.name))
        self.function_depth += 1
        self.generic_visit(node)
        self.function_depth -= 1
        self.context.pop()

    def visit_FunctionDef(self, node):
        self._visit_function(node)

    def visit_AsyncFunctionDef(self, node):
        self._visit_function(node, is_async=True)


for module in module_data:
    DefinitionVisitor(module).visit(module_data[module]["tree"])

print(f"Parsed {len(module_data)} modules, {len(classes)} classes, and {len(definitions)} functions/methods.")

Parsed 49 modules, 32 classes, and 717 functions/methods.


## 2. Resolve internal calls

Calls are resolved conservatively. Direct module functions, imported functions, `self.method()`, `cls.method()`, `super().method()`, and explicit class methods are recognized. Ambiguous object-method calls remain unresolved rather than being assigned to a potentially incorrect implementation.

In [9]:
def resolve_relative_module(current_module, imported_module, level):
    current = module_data[current_module]
    base = current_module.split(".") if current["is_package"] else current_module.split(".")[:-1]
    if level > 1:
        base = base[:-(level - 1)]
    if imported_module:
        base.extend(imported_module.split("."))
    return ".".join(base)


imports = {}
for module, data in module_data.items():
    aliases = {}
    for node in ast.walk(data["tree"]):
        if isinstance(node, ast.Import):
            for alias in node.names:
                aliases[alias.asname or alias.name.split(".")[0]] = alias.name
        elif isinstance(node, ast.ImportFrom):
            imported_module = node.module or ""
            if node.level:
                imported_module = resolve_relative_module(module, imported_module, node.level)
            for alias in node.names:
                if alias.name != "*":
                    aliases[alias.asname or alias.name] = imported_module + "." + alias.name
    imports[module] = aliases


module_functions = {}
class_methods = {}
simple_names = defaultdict(list)
for function_id, function in definitions.items():
    simple_names[function["name"]].append(function_id)
    if function["owner_class"]:
        class_methods[(function["owner_class"], function["name"])] = function_id
    elif function["kind"] in ("function", "async_function"):
        module_functions[(function["module"], function["name"])] = function_id


def resolve_class_name(name, module):
    direct = module + "." + name
    if direct in classes:
        return direct
    imported = imports[module].get(name)
    if imported in classes:
        return imported
    return None


def find_method(class_id, method_name, visited=None):
    if class_id is None:
        return None
    if visited is None:
        visited = set()
    if class_id in visited:
        return None
    visited.add(class_id)
    direct = class_methods.get((class_id, method_name))
    if direct:
        return direct
    class_data = classes.get(class_id, {})
    for base in class_data.get("bases", []):
        base_root = base.split(".")[0]
        imported = imports[class_data["module"]].get(base_root)
        if imported:
            suffix = base.split(".", 1)[1] if "." in base else ""
            base_id = imported + ("." + suffix if suffix else "")
        else:
            base_id = resolve_class_name(base, class_data["module"])
        resolved = find_method(base_id, method_name, visited)
        if resolved:
            return resolved
    return None


def attribute_parts(node):
    parts = []
    while isinstance(node, ast.Attribute):
        parts.append(node.attr)
        node = node.value
    if isinstance(node, ast.Name):
        parts.append(node.id)
        return list(reversed(parts))
    return None


def resolve_exact_path(path):
    if path in definitions:
        return path
    constructor = path + ".__init__"
    if constructor in definitions:
        return constructor
    return None


def resolve_call(call, caller):
    function = call.func
    module = caller["module"]
    owner_class = caller["owner_class"]

    if isinstance(function, ast.Name):
        direct = module_functions.get((module, function.id))
        if direct:
            return direct, "same_module"
        imported = imports[module].get(function.id)
        resolved = resolve_exact_path(imported) if imported else None
        if resolved:
            return resolved, "import"
        class_id = resolve_class_name(function.id, module)
        if class_id:
            constructor = find_method(class_id, "__init__")
            if constructor:
                return constructor, "constructor"
        return None, None

    if isinstance(function, ast.Attribute):
        if isinstance(function.value, ast.Name) and function.value.id in ("self", "cls"):
            resolved = find_method(owner_class, function.attr)
            if resolved:
                return resolved, "class_method"
        if isinstance(function.value, ast.Call) and isinstance(function.value.func, ast.Name) and function.value.func.id == "super":
            class_data = classes.get(owner_class, {})
            for base in class_data.get("bases", []):
                base_id = resolve_class_name(base, module)
                resolved = find_method(base_id, function.attr)
                if resolved:
                    return resolved, "super_method"
        parts = attribute_parts(function)
        if parts:
            root = parts[0]
            imported = imports[module].get(root)
            if imported:
                resolved = resolve_exact_path(".".join([imported] + parts[1:]))
                if resolved:
                    return resolved, "qualified_import"
            class_id = resolve_class_name(root, module)
            if class_id and len(parts) == 2:
                resolved = find_method(class_id, parts[1])
                if resolved:
                    return resolved, "explicit_class"
    return None, None


class FunctionCallVisitor(ast.NodeVisitor):
    def __init__(self, root):
        self.root = root
        self.calls = []

    def visit_FunctionDef(self, node):
        if node is self.root:
            for statement in node.body:
                self.visit(statement)

    def visit_AsyncFunctionDef(self, node):
        self.visit_FunctionDef(node)

    def visit_ClassDef(self, node):
        return

    def visit_Call(self, node):
        self.calls.append(node)
        self.generic_visit(node)


edge_calls = defaultdict(list)
unresolved_calls = defaultdict(list)
for caller_id, function in definitions.items():
    visitor = FunctionCallVisitor(definition_asts[caller_id])
    visitor.visit(definition_asts[caller_id])
    for call in visitor.calls:
        expression = ast.unparse(call.func)
        callee_id, resolution = resolve_call(call, function)
        callsite = {
            "file": function["file"],
            "line": call.lineno,
            "expression": expression,
            "resolution": resolution,
        }
        if callee_id:
            edge_calls[(caller_id, callee_id)].append(callsite)
        else:
            unresolved_calls[caller_id].append(callsite)

edges = []
for (caller_id, callee_id), callsites in edge_calls.items():
    edges.append({
        "id": caller_id + " -> " + callee_id,
        "source": caller_id,
        "target": callee_id,
        "count": len(callsites),
        "callsites": callsites,
    })

incoming = Counter(edge["target"] for edge in edges)
outgoing = Counter(edge["source"] for edge in edges)
for function_id, function in definitions.items():
    function["incoming"] = incoming[function_id]
    function["outgoing"] = outgoing[function_id]
    function["unresolved"] = unresolved_calls[function_id]

print(f"Resolved {sum(len(value) for value in edge_calls.values())} call sites into {len(edges)} directed edges.")
print(f"Retained {sum(len(value) for value in unresolved_calls.values())} unresolved or external call sites.")

Resolved 953 call sites into 668 directed edges.
Retained 6458 unresolved or external call sites.


## 3. Export graph data

In [10]:
module_edges = defaultdict(int)
for edge in edges:
    source_module = definitions[edge["source"]]["module"]
    target_module = definitions[edge["target"]]["module"]
    if source_module != target_module:
        module_edges[(source_module, target_module)] += edge["count"]

module_counts = Counter(function["module"] for function in definitions.values())
git_commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=scope_repo_path, capture_output=True, text=True, check=True).stdout.strip()
git_dirty  = bool(subprocess.run(["git", "status", "--porcelain"], cwd=scope_repo_path, capture_output=True, text=True, check=True).stdout.strip())

graph_data = {
    "metadata": {
        "scope_core_path": scope_core_path,
        "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "git_commit": git_commit,
        "working_tree_dirty": git_dirty,
        "module_count": len(module_data),
        "class_count": len(classes),
        "function_count": len(definitions),
        "edge_count": len(edges),
    },
    "nodes": list(definitions.values()),
    "edges": edges,
    "modules": [
        {
            "id": module,
            "label": module.removeprefix("scope.") or "scope",
            "file": data["relative_file"],
            "function_count": module_counts[module],
        }
        for module, data in module_data.items()
    ],
    "module_edges": [
        {"source": source, "target": target, "count": count}
        for (source, target), count in module_edges.items()
    ],
}

with open(json_output, "w", encoding="utf-8") as file:
    json.dump(graph_data, file, indent=2)

print(f"Graph data written to {json_output}")

Graph data written to /Users/sergivela/Documents/SCOPE/Program/Scope/docs/development/scope_core_graph/scope_core_graph.json


## 4. Generate the interactive map

The HTML uses `vis-network` from a public CDN. The graph data itself is embedded in the file; an internet connection is only required to load the visualization library.

In [11]:
graph_json = json.dumps(graph_data).replace("</", "<" + chr(92) + "/")

html = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>SCOPE Core Call Graph</title>
<script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
<style>
:root { --ink:#17221d; --paper:#f5f2e8; --panel:#fffdf6; --line:#c8c2b2; --accent:#bb4d2f; }
* { box-sizing:border-box; }
body { margin:0; height:100vh; display:flex; flex-direction:column; overflow:hidden; color:var(--ink); background:var(--paper); font-family:Georgia, 'Times New Roman', serif; }
header { padding:14px 18px 10px; border-bottom:2px solid var(--ink); background:#e8e1ce; }
h1 { margin:0 0 4px; font-size:24px; font-weight:700; }
.subtitle { font-size:13px; color:#4f594f; }
.toolbar { display:flex; flex-wrap:wrap; gap:8px 14px; align-items:end; padding:10px 14px; border-bottom:1px solid var(--line); background:var(--panel); }
.control { display:flex; flex-direction:column; gap:3px; font-size:11px; font-weight:700; letter-spacing:.04em; text-transform:uppercase; }
select,input,button { min-height:30px; border:1px solid #817b6d; border-radius:3px; background:white; color:var(--ink); padding:4px 8px; font:13px Georgia,serif; }
#search { width:360px; }
button { cursor:pointer; background:var(--ink); color:white; border-color:var(--ink); }
.checks { display:flex; gap:12px; align-items:center; min-height:30px; font-size:12px; }
.checks label { display:flex; gap:4px; align-items:center; }
.layout { display:grid; grid-template-columns:minmax(0,1fr) 360px; flex:1; min-height:0; overflow:hidden; }
#graph-shell { position:relative; min-width:0; min-height:0; height:100%; overflow:hidden; border-right:1px solid var(--line); background:#fbfaf5; }
#graph { width:100%; height:100%; }
#graph-status { position:absolute; z-index:5; top:14px; left:50%; transform:translateX(-50%); padding:7px 11px; border:1px solid var(--line); border-radius:3px; background:rgba(255,253,246,.94); font-size:12px; pointer-events:none; }
#graph-status::before { content:''; display:inline-block; width:10px; height:10px; margin-right:7px; border:2px solid #c8c2b2; border-top-color:var(--accent); border-radius:50%; vertical-align:-2px; animation:spin .7s linear infinite; }
#graph-status.hidden { display:none; }
@keyframes spin { to { transform:rotate(360deg); } }
#details { overflow:auto; padding:16px; background:var(--panel); }
#details h2 { margin:0 0 8px; font-size:18px; overflow-wrap:anywhere; }
#details h3 { margin:18px 0 6px; font-size:14px; border-bottom:1px solid var(--line); padding-bottom:3px; }
#details p,#details li { font-size:12px; line-height:1.45; }
#details ul { margin:5px 0; padding-left:18px; }
.path { font-family:Menlo,Consolas,monospace; font-size:11px; overflow-wrap:anywhere; }
.focus-link { color:#8e321d; cursor:pointer; text-decoration:underline; }
.legend { display:flex; flex-wrap:wrap; gap:10px; margin-top:8px; font-size:11px; }
.swatch { display:inline-block; width:10px; height:10px; border-radius:50%; margin-right:3px; }
@media (max-width:900px) { body{height:auto;min-height:100vh;overflow:auto}.layout{grid-template-columns:1fr;flex:none;height:auto;overflow:visible}#graph-shell{height:70vh;border-right:0;border-bottom:1px solid var(--line)}#details{height:40vh}#search{width:250px} }
</style>
</head>
<body>
<header>
  <h1>SCOPE Core Call Graph</h1>
  <div class="subtitle" id="summary"></div>
  <div class="legend"><span><i class="swatch" style="background:#31708e"></i>method</span><span><i class="swatch" style="background:#c86b36"></i>function</span><span><i class="swatch" style="background:#6f8064"></i>nested</span><span><i class="swatch" style="background:#8f6ca8"></i>async</span></div>
</header>
<div class="toolbar">
  <label class="control">View<select id="view"><option value="overview">Module overview</option><option value="module">Functions in selected module</option><option value="all">Entire core</option></select></label>
  <label class="control">Selected module<select id="module"></select></label>
  <label class="control">Global function search<input id="search" list="functions" placeholder="Qualified function name"><datalist id="functions"></datalist></label>
  <label class="control">Depth<select id="depth" title="Number of caller/callee steps shown around a focused function"><option>1</option><option selected>2</option><option>3</option></select></label>
  <button id="focus">Focus</button>
  <div class="checks"><label><input type="checkbox" id="private" checked> private</label><label><input type="checkbox" id="dunder"> dunder</label></div>
</div>
<main class="layout"><div id="graph-shell"><div id="graph-status" role="status" aria-live="polite">Preparing graph...</div><div id="graph"></div></div><aside id="details"><h2>How to navigate</h2><p><b>Module browsing:</b> choose <i>Functions in selected module</i>, then select a module. Double-clicking a module in the overview does the same.</p><p><b>Function search:</b> select any qualified function globally and press <b>Focus</b>. This search is independent of the selected module.</p><p><b>Depth:</b> 1 shows the selected function and its immediate callers/callees. Depth 2 also shows their immediate connections, providing broader context without displaying the entire package.</p><p>Click any function to inspect its source location, resolved calls, and unresolved or external calls.</p></aside></main>
<script>
const DATA = __GRAPH_DATA__;
const nodeById = Object.fromEntries(DATA.nodes.map(node => [node.id,node]));
const moduleById = Object.fromEntries(DATA.modules.map(node => [node.id,node]));
const incoming = {}, outgoing = {};
for (const edge of DATA.edges) { (outgoing[edge.source] ||= []).push(edge); (incoming[edge.target] ||= []).push(edge); }
const graphElement=document.getElementById('graph'), graphStatus=document.getElementById('graph-status'), details=document.getElementById('details');
let network=null, currentSeed=null, renderToken=0;
const revision=DATA.metadata.git_commit.slice(0,8)+(DATA.metadata.working_tree_dirty?' + local changes':'');
document.getElementById('summary').textContent=`Snapshot ${revision} | ${DATA.metadata.module_count} modules | ${DATA.metadata.class_count} classes | ${DATA.metadata.function_count} functions and methods | ${DATA.metadata.edge_count} resolved call relations`;
const moduleSelect=document.getElementById('module');
for (const module of DATA.modules.sort((a,b)=>a.id.localeCompare(b.id))) { const option=document.createElement('option'); option.value=module.id; option.textContent=module.id; moduleSelect.appendChild(option); }
const datalist=document.getElementById('functions');
for (const node of DATA.nodes.sort((a,b)=>a.id.localeCompare(b.id))) { const option=document.createElement('option'); option.value=node.id; datalist.appendChild(option); }
function allowed(node, force=false) { if(force)return true; if(!document.getElementById('private').checked && node.private)return false; if(!document.getElementById('dunder').checked && node.dunder)return false; return true; }
function groupFor(node) { if(node.kind.startsWith('async_'))return 'async'; return node.kind; }
function visualNode(node, force=false) { return {id:node.id,label:node.label,group:groupFor(node),value:Math.max(2,2+node.incoming+node.outgoing),title:`${node.id}\n${node.file}:${node.line}`,font:{size:force?17:13}}; }
function visualEdge(edge) { return {id:edge.id,from:edge.source,to:edge.target,arrows:'to',value:edge.count,title:edge.callsites.map(site=>`${site.file}:${site.line}  ${site.expression}`).join('\n')}; }
function draw(nodes,edges,selectedId=null) { const token=++renderToken; graphStatus.textContent=`Building graph: ${nodes.length} nodes and ${edges.length} connections...`; graphStatus.classList.remove('hidden'); setTimeout(()=>{if(token!==renderToken)return;if(network)network.destroy();const datasets={nodes:new vis.DataSet(nodes),edges:new vis.DataSet(edges)};const options={groups:{method:{color:'#31708e'},function:{color:'#c86b36'},nested:{color:'#6f8064'},async:{color:'#8f6ca8'},async_method:{color:'#8f6ca8'},async_function:{color:'#8f6ca8'},async_nested:{color:'#8f6ca8'},module:{color:'#b9a44c',shape:'box'}},nodes:{shape:'dot',borderWidth:1,font:{face:'Georgia',color:'#17221d'}},edges:{color:{color:'#918c80',highlight:'#bb4d2f'},smooth:{type:'dynamic'},scaling:{min:1,max:5}},interaction:{hover:true,navigationButtons:true,keyboard:true},physics:{solver:'forceAtlas2Based',forceAtlas2Based:{gravitationalConstant:-55,springLength:115,springConstant:.04},stabilization:{iterations:350}}};network=new vis.Network(graphElement,datasets,options);if(selectedId)network.selectNodes([selectedId]);const clearStatus=()=>graphStatus.classList.add('hidden');network.once('stabilized',clearStatus);setTimeout(clearStatus,5000);network.on('click',params=>{if(params.nodes.length)showDetails(params.nodes[0]);});network.on('doubleClick',params=>{if(params.nodes.length&&String(params.nodes[0]).startsWith('module::'))showModule(String(params.nodes[0]).slice(8));});},30); }
function showOverview() { currentSeed=null; const nodes=DATA.modules.map(module=>({id:'module::'+module.id,label:`${module.label}\n(${module.function_count})`,group:'module',value:Math.max(3,module.function_count),title:`${module.id}\n${module.file}`})); const edges=DATA.module_edges.map((edge,index)=>({id:'me'+index,from:'module::'+edge.source,to:'module::'+edge.target,arrows:'to',value:edge.count,title:`${edge.count} resolved calls`})); draw(nodes,edges); details.innerHTML='<h2>Module overview</h2><p>Node size represents the number of functions and methods. Edge width represents resolved calls between modules. Double-click a module to inspect its contents.</p>'; }
function filteredNodes(nodes,forced=[]) { const force=new Set(forced); return nodes.filter(node=>allowed(node,force.has(node.id))); }
function showModule(module) { document.getElementById('view').value='module'; moduleSelect.disabled=false; moduleSelect.value=module; const selected=filteredNodes(DATA.nodes.filter(node=>node.module===module)); const ids=new Set(selected.map(node=>node.id)); const edges=DATA.edges.filter(edge=>ids.has(edge.source)&&ids.has(edge.target)); draw(selected.map(node=>visualNode(node)),edges.map(visualEdge)); details.innerHTML=`<h2>${escapeHtml(module)}</h2><p>${selected.length} visible functions and methods.</p><p class="path">${escapeHtml(moduleById[module].file)}</p><p>Click a function to inspect it, or use the global function search to open a caller/callee neighborhood.</p>`; }
function showAll() { currentSeed=null; const selected=filteredNodes(DATA.nodes); const ids=new Set(selected.map(node=>node.id)); const edges=DATA.edges.filter(edge=>ids.has(edge.source)&&ids.has(edge.target)); draw(selected.map(node=>visualNode(node)),edges.map(visualEdge)); details.innerHTML=`<h2>Entire core</h2><p>${selected.length} visible functions and methods. This view can be dense; use search and Focus for local inspection.</p>`; }
function neighborhood(seed,depth) { const found=new Set([seed]), frontier=[seed]; for(let level=0;level<depth;level++){const next=[];for(const id of frontier){for(const edge of [...(incoming[id]||[]),...(outgoing[id]||[])]){const other=edge.source===id?edge.target:edge.source;if(!found.has(other)){found.add(other);next.push(other);}}}frontier.splice(0,frontier.length,...next);}return found;}
function focusFunction(id) { if(!nodeById[id]){graphStatus.textContent='Select an exact function from the search suggestions.';graphStatus.classList.remove('hidden');details.innerHTML='<h2>Function not found</h2><p>Select a qualified function name from the global search suggestions and press <b>Focus</b>.</p>';return;} currentSeed=id; const ids=neighborhood(id,Number(document.getElementById('depth').value)); const selected=filteredNodes(DATA.nodes.filter(node=>ids.has(node.id)),[id]); const visible=new Set(selected.map(node=>node.id)); const edges=DATA.edges.filter(edge=>visible.has(edge.source)&&visible.has(edge.target)); draw(selected.map(node=>visualNode(node,node.id===id)),edges.map(visualEdge),id); showDetails(id); }
function escapeHtml(value) { return String(value).replace(/[&<>"']/g,char=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#039;'}[char])); }
function focusLink(id) { return `<span class="focus-link" data-id="${escapeHtml(id)}">${escapeHtml(id)}</span>`; }
function showDetails(id) { if(String(id).startsWith('module::')){const module=String(id).slice(8),data=moduleById[module];details.innerHTML=`<h2>${escapeHtml(module)}</h2><p>${data.function_count} functions and methods.</p><p class="path">${escapeHtml(data.file)}</p><p>Double-click this node to open the module.</p>`;return;} const node=nodeById[id];if(!node)return; const callers=incoming[id]||[],callees=outgoing[id]||[]; const list=(items,direction)=>items.length?'<ul>'+items.map(edge=>{const other=direction==='in'?edge.source:edge.target;const sites=edge.callsites.map(site=>`${site.file}:${site.line}`).join(', ');return `<li>${focusLink(other)}<br><span class="path">${escapeHtml(sites)}</span></li>`;}).join('')+'</ul>':'<p>None resolved.</p>'; const unresolved=node.unresolved.slice(0,30).map(site=>`<li><span class="path">${escapeHtml(site.expression)} - ${escapeHtml(site.file)}:${site.line}</span></li>`).join(''); details.innerHTML=`<h2>${escapeHtml(node.id)}</h2><p><b>${escapeHtml(node.kind)}</b></p><p class="path">${escapeHtml(node.file)}:${node.line}-${node.end_line}</p><p>The selected node and its connecting edges are highlighted. Click a listed caller or callee to refocus the map.</p>${node.doc?`<p>${escapeHtml(node.doc)}</p>`:''}<h3>Called by (${callers.length})</h3>${list(callers,'in')}<h3>Calls (${callees.length})</h3>${list(callees,'out')}<h3>Unresolved/external (${node.unresolved.length})</h3>${unresolved?'<ul>'+unresolved+'</ul>':'<p>None.</p>'}`; for(const link of details.querySelectorAll('.focus-link'))link.addEventListener('click',()=>{document.getElementById('search').value=link.dataset.id;focusFunction(link.dataset.id);}); }
function updateModuleControl() { moduleSelect.disabled=document.getElementById('view').value!=='module'; }
function refreshView() { updateModuleControl(); const view=document.getElementById('view').value;if(view==='overview')showOverview();else if(view==='module')showModule(moduleSelect.value);else showAll(); }
document.getElementById('focus').addEventListener('click',()=>focusFunction(document.getElementById('search').value.trim()));
document.getElementById('search').addEventListener('keydown',event=>{if(event.key==='Enter')focusFunction(event.target.value.trim());});
document.getElementById('view').addEventListener('change',refreshView); moduleSelect.addEventListener('change',()=>{if(document.getElementById('view').value==='module')showModule(moduleSelect.value);});
document.getElementById('depth').addEventListener('change',()=>{if(currentSeed)focusFunction(currentSeed);}); document.getElementById('private').addEventListener('change',()=>currentSeed?focusFunction(currentSeed):refreshView()); document.getElementById('dunder').addEventListener('change',()=>currentSeed?focusFunction(currentSeed):refreshView());
updateModuleControl(); showOverview();
</script>
</body></html>'''.replace('__GRAPH_DATA__', graph_json)

with open(html_output, "w", encoding="utf-8") as file:
    file.write(html)

print(f"Interactive graph written to {html_output}")

Interactive graph written to /Users/sergivela/Documents/SCOPE/Program/Scope/docs/development/scope_core_graph/scope_core_graph.html


## 5. Preview

In [12]:
display(IFrame(html_output, width="100%", height=760))